# Bing Grounding Tool

Creates a persistent Foundry agent with the **Bing Grounding** tool enabled.
When the user asks about recent events the agent searches Bing automatically.

In [ ]:
%pip install azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    BingGroundingTool,
    BingGroundingSearchToolParameters,
    BingGroundingSearchConfiguration,
)
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from repo root

endpoint = os.environ["AZURE_FOUNDRY_ENDPOINT"]
bing_connection_name = os.environ["BING_CONNECTION_ID"]

client = AIProjectClient(endpoint=endpoint, credential=DefaultAzureCredential())
openai_client = client.get_openai_client()

# Resolve the Bing Grounding connection ID by name from the project's connections
bing_connection_id = next(
    c.id for c in client.connections.list() if c.name == bing_connection_name
)
print(f"Client ready. Bing connection: {bing_connection_name}")

In [ ]:
# Define the Bing Grounding tool and create a persistent Foundry v2 (Prompt) agent
bing_tool = BingGroundingTool(
    bing_grounding=BingGroundingSearchToolParameters(
        search_configurations=[
            BingGroundingSearchConfiguration(project_connection_id=bing_connection_id),
        ],
    ),
)

agent = client.agents.create_version(
    agent_name="GroundedSearchAgent",
    definition=PromptAgentDefinition(
        model="gpt-4.1",
        instructions="You answer questions using Bing search to find current, accurate information.",
        tools=[bing_tool],
    ),
)
print(f"Agent created: {agent.name} (version {agent.version})")

In [ ]:
# Ask a question that requires live information — stream the agent's response
print("--- Agent Response ---\n")

stream = openai_client.responses.create(
    stream=True,
    input="Who is the president of the United States?",
    extra_body={
        "agent_reference": {
            "name": agent.name,
            "version": agent.version,
            "type": "agent_reference",
        }
    },
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
print()

In [ ]:
# Cleanup: delete the agent (all versions) when done
client.agents.delete(agent_name=agent.name)
print("\nAgent deleted.")